# E4-lite — E2 결과는 시드를 바꿔도 유지되는가

## 왜 하나

E2(문장 앞부분 학습)는 사전 기준을 충족했다. 4토큰마다 재판정할 때 재현율 0.494, 탐지 지연 6.5토큰이었다.
하지만 **학습을 한 번(시드 42, 1에폭)만** 돌렸고, 학습 곡선도 끝까지 오르는 중이었다(검증 앞부분 AUROC 0.839 → 0.853).
결과가 우연인지, 그리고 더 학습하면 좋아지는지를 확인해야 중간보고에서 이 결과를 주장할 수 있다.

## 무엇을 하나

- 시드 **3개(42, 7, 2026)** × **2에폭**으로 다시 학습한다. 시드는 앞부분 자르는 위치, 학습 순서, 드롭아웃을 모두 바꾼다.
- 그 밖의 모든 설정(데이터, 정답 정의, 가중치, 근거 검색, 평가 코드)은 E2와 같다. E2 노트북에서 코드를 그대로 가져왔다.
- 체크포인트는 2에폭 동안 6번 검증 분할로 평가해 가장 좋은 것을 고른다.
- 비교용으로 E2 원본 모델(`student_prefix`, 시드 42·1에폭)과 기준선(`student_vark`)도 같은 질의로 다시 판정한다.

## 사전 정의 기준 (결과 보기 전, SSOT §6에 고정)

- **안정성(주 기준)**: 시드 3개 **모두** E2 주 기준을 충족한다. E2 주 기준은 ARL₀=100에서 s=4 또는 s=2 중 하나라도 재현율 ≥ 0.474이면서 지연 중앙값 ≤ 9토큰인 것이다.
- **평균**: s=4에서 시드 평균 재현율 ≥ 0.474, 시드 평균 지연 중앙값 ≤ 9토큰
- **보고**: 시드 간 표준편차와 2에폭 효과(두 번째 에폭에서 체크포인트가 선택된 시드 수)는 판정 없이 보고한다.
- 시드 3개 중 하나라도 미달하면 "E2 결과는 시드에 민감"으로 기록하고, 중간보고에서는 시드 평균만 주장한다.

> 런타임 **GPU(T4)**. 시드당 학습 약 20~25분 → 전체 약 1.5시간. 시드마다 모델을 드라이브 `selfproject/student_prefix_seeds/`에 저장하므로,
> 중간에 끊겨도 다시 실행하면 이미 끝난 시드는 건너뛴다.


## 1. 환경 준비

In [ ]:
!pip -q install pysbd sentence-transformers tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'

import json, time, random, os, gc, copy
import numpy as np, pandas as pd, torch, pysbd, tiktoken
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/selfproject' if USE_DRIVE else '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STUDENT_DIR = os.path.join(OUT_DIR, 'student_vark')
SAVE_ROOT = os.path.join(OUT_DIR, 'student_prefix_seeds')
E2_DIR = os.path.join(OUT_DIR, 'student_prefix')
assert os.path.isdir(STUDENT_DIR), f'{STUDENT_DIR} 없음'
print('device', DEVICE)

## 2. 데이터 (E2와 같음)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
# 학습 응답: S2/S2-b의 gold-large와 같은 구성 (teacher 학습 510 + 추가 2,490). dev·검증과 겹치지 않음을 확인한다.
teacher_train = teacher_resp[len(val):]
used = dev_ids | {r['id'] for r in teacher_resp}
gl_extra = rng.sample([r for r in rest if r['id'] not in used], 3000 - len(teacher_train))
train = teacher_train + gl_extra
tr_ids = {r['id'] for r in train}
assert not (tr_ids & dev_ids) and not (tr_ids & {r['id'] for r in val}), '학습과 dev/검증 겹침'
print(f'학습 {len(train)} 응답 (dev·검증과 겹침 없음)')

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

In [ ]:
train_items = to_items(train)
print(f'학습: 응답 {len(train_items)}, 문장 {sum(len(x["sents"]) for x in train_items):,}')

## 3. 앞부분 학습 데이터 함수 (E2와 같음 — 시드마다 다시 만든다)

In [ ]:
MIN_PREFIX, NEAR_AFTER, W_NEAR = 2, 8, 2.0

def sent_onset(x, sn):
    # 문장 안 첫 환각 토큰 번호 (없으면 None). 응답 전체의 첫 발생이 아니라 '이 문장 안'의 첫 발생
    spans = [(l['start'], l['end']) for l in x['labels']]
    for t in range(sn['t0'], sn['t1'] + 1):
        if any(min(x['te'][t], b) > max(x['cs'][t], a) for a, b in spans):
            return t
    return None

def prefix_examples(x, rng):
    out = []
    for sn in x['sents']:
        t0, t1 = sn['t0'], sn['t1']
        if t1 - t0 + 1 < MIN_PREFIX: continue
        on = sent_onset(x, sn)
        cand = list(range(t0 + MIN_PREFIX - 1, t1))              # 문장 끝 제외한 자르는 위치
        cuts = {t1}
        if on is None:
            cuts |= set(rng.sample(cand, min(3, len(cand))))
        else:
            cuts |= set(rng.sample(cand, min(2, len(cand))))
            near = [c for c in range(max(t0 + MIN_PREFIX - 1, on - 1), min(t1, on + NEAR_AFTER + 1))]
            cuts |= set(rng.sample(near, min(2, len(near))))
        for c in sorted(cuts):
            text = x['text'][sn['a']:min(x['te'][c], sn['b'])].strip()
            if not text: continue
            y = int(on is not None and on <= c)
            w = W_NEAR if (y and c - on <= NEAR_AFTER) else 1.0
            out.append({'ref_key': x['ref_key'], 'text': text, 'y': y, 'w': w, 'full': c == t1})
    return out

# prep()에 labels·cs가 필요하므로 원본 응답에서 붙인다
by_id = {str(r['id']): r for r in train}
for x in train_items:
    r = by_id[x['id']]
    x['labels'] = r.get('labels') or []
    text = x['text']
    x['cs'] = [a + (len(text[a:b]) - len(text[a:b].lstrip())) if text[a:b].strip() else a
               for a, b in zip(x['ts'], x['te'])]

prng = random.Random(SEED)
train_ex = pd.DataFrame([e for x in train_items for e in prefix_examples(x, prng)])
print(f'앞부분 예시 {len(train_ex):,}개 | 양성 {train_ex.y.mean():.1%} | 완성 문장 {train_ex.full.mean():.1%} | '
      f'발생 직후 가중 {(train_ex.w > 1).mean():.1%}')

## 4. 근거 검색과 판정 함수 (E2와 같음)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

retriever = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
K, MAX_LEN, CHUNK_WORDS, BS = 8, 512, 40, 32

chunk_cache = {}
def get_chunks(key):
    if key not in chunk_cache:
        w = REFS[key].split()
        ch = [' '.join(w[i:i + CHUNK_WORDS]) for i in range(0, max(len(w), 1), CHUNK_WORDS)] or [REFS[key]]
        chunk_cache[key] = (ch, retriever.encode(ch, convert_to_tensor=True, normalize_embeddings=True,
                                                 batch_size=128, show_progress_bar=False))
    return chunk_cache[key]

def contexts(keys, texts, bs=256):
    # 텍스트마다 근거 top-K 조각을 원문 순서로 이어 붙인 문자열
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            top = torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()
            out.append(' ... '.join(ch[t] for t in sorted(top)))
    return out

@torch.no_grad()
def score(tok, model, keys, texts, bs=128):
    model.eval()
    out = np.zeros(len(texts), dtype=np.float32)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        ctx = contexts([keys[j] for j in idx], [texts[j] for j in idx])
        enc = tok(ctx, [texts[j] for j in idx], truncation='only_first', max_length=MAX_LEN,
                  padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
            out[idx] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
    return out

## 5. 판정 지점과 검증 정답 (E2와 같음)

In [ ]:
def points(item, s):
    pts = []
    for sn in item['sents']:
        for t in range(sn['t0'], sn['t1'] + 1):
            k = t - sn['t0'] + 1
            if (s is not None and k % s == 0) or t == sn['t1']:
                pts.append((t, item['text'][sn['a']:min(item['te'][t], sn['b'])].strip()))
    return pts

QUERY_ID, queries = {}, []
def qid(key, text):
    if (key, text) not in QUERY_ID:
        QUERY_ID[(key, text)] = len(queries); queries.append((key, text))
    return QUERY_ID[(key, text)]

for x in dev_items + val_items:
    x['pts2'] = [(t, qid(x['ref_key'], txt)) for t, txt in points(x, 2) if txt]
Q_KEYS, Q_TEXTS = [q[0] for q in queries], [q[1] for q in queries]

# 검증 분할 앞부분의 정답 (체크포인트 선택용): 판정 지점 t까지 이 문장 안에 환각이 시작됐는가
def point_labels(items):
    ids, ys = [], []
    for x in items:
        spans = [(l['start'], l['end']) for l in x['labels']]
        for t, q in x['pts2']:
            a = x['sent_of'][t]
            ys.append(int(any(min(x['te'][u], b) > max(x['cs'][u], s0) for u in range(a, t + 1) for s0, b in spans)))
            ids.append(q)
    return np.array(ids), np.array(ys)

by_all = {str(r['id']): r for r in dev + val}
for x in dev_items + val_items:
    x['labels'] = by_all[x['id']].get('labels') or []
    text = x['text']
    x['cs'] = [a + (len(text[a:b]) - len(text[a:b].lstrip())) if text[a:b].strip() else a
               for a, b in zip(x['ts'], x['te'])]
VAL_Q, VAL_Y = point_labels(val_items)
print(f'판정 질의 {len(queries):,}개 | 검증 판정 지점 {len(VAL_Q):,}개 (양성 {VAL_Y.mean():.1%})')

## 6. 학습 함수 (E2와 같음)

In [ ]:
def val_auroc(tok, model):
    uq = np.unique(VAL_Q)
    s = score(tok, model, [Q_KEYS[i] for i in uq], [Q_TEXTS[i] for i in uq])
    m = dict(zip(uq, s))
    return roc_auc_score(VAL_Y, [m[q] for q in VAL_Q])

def train_prefix(init_dir, epochs=1, lr=2e-5, n_evals=3):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    tok = AutoTokenizer.from_pretrained(init_dir)
    model = AutoModelForSequenceClassification.from_pretrained(init_dir).to(DEVICE)
    base = val_auroc(tok, model)
    print(f'  시작점 검증 앞부분 AUROC {base:.3f}')
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    n_step = epochs * ((len(train_ex) + BS - 1) // BS)
    sched = get_linear_schedule_with_warmup(opt, max(1, int(0.06 * n_step)), n_step)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
    eval_at = set(np.linspace(0, n_step, n_evals + 1, dtype=int)[1:])
    best, best_state, step, hist = base, copy.deepcopy(model.state_dict()), 0, [(0, base)]
    for ep in range(epochs):
        order = np.random.permutation(len(train_ex))
        for i in range(0, len(order), BS):
            model.train()
            b = train_ex.iloc[order[i:i + BS]]
            enc = tok(b.ctx.tolist(), b.text.tolist(), truncation='only_first', max_length=MAX_LEN,
                      padding=True, return_tensors='pt').to(DEVICE)
            y = torch.tensor(b.y.values, dtype=torch.float32, device=DEVICE)
            w = torch.tensor(b.w.values, dtype=torch.float32, device=DEVICE)
            with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
                logit = model(**enc).logits.squeeze(-1)
            loss = (torch.nn.functional.binary_cross_entropy_with_logits(logit.float(), y, reduction='none') * w).mean()
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            step += 1
            if step in eval_at:
                va = val_auroc(tok, model); hist.append((step, va))
                print(f'  step {step}/{n_step}  loss {loss.item():.3f}  검증 앞부분 AUROC {va:.3f}')
                if va > best:
                    best, best_state = va, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return tok, model, best, hist

## 7. 시드별 학습 — 끝난 시드는 건너뜀

In [ ]:
SEEDS, EPOCHS, N_EVALS = [42, 7, 2026], 2, 6
os.makedirs(SAVE_ROOT, exist_ok=True)
train_log = []
for seed in SEEDS:
    out = os.path.join(SAVE_ROOT, f'seed{seed}_ep{EPOCHS}')
    if os.path.exists(os.path.join(out, 'config.json')):
        print(f'[시드 {seed}] 이미 학습됨 → 건너뜀'); continue
    SEED = seed                                     # train_prefix가 전역 SEED로 난수를 고정한다
    prng = random.Random(seed)
    train_ex = pd.DataFrame([e for x in train_items for e in prefix_examples(x, prng)])
    train_ex['ctx'] = contexts(train_ex.ref_key.tolist(), train_ex.text.tolist())
    print(f'[시드 {seed}] 앞부분 예시 {len(train_ex):,}개, 양성 {train_ex.y.mean():.1%}')
    t0 = time.perf_counter()
    tok_s, model_s, best_va, hist = train_prefix(STUDENT_DIR, epochs=EPOCHS, n_evals=N_EVALS)
    best_step = max(hist, key=lambda h: h[1])[0]
    n_step = hist[-1][0]
    tok_s.save_pretrained(out); model_s.save_pretrained(out)
    pd.DataFrame(hist, columns=['step', 'val_prefix_auroc']).to_csv(os.path.join(out, 'hist.csv'), index=False)
    print(f'[시드 {seed}] {(time.perf_counter()-t0)/60:.1f}분 | 최고 검증 앞부분 AUROC {best_va:.3f} '
          f'(step {best_step}/{n_step}, {"2" if best_step > n_step // 2 else "1"}번째 에폭)')
    del model_s; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

## 8. 판정 — 기준선, E2 원본, 시드별 모델을 같은 질의에 적용

In [ ]:
SC = {}
def run(name, path):
    tok = AutoTokenizer.from_pretrained(path)
    m = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE)
    SC[name] = score(tok, m, Q_KEYS, Q_TEXTS)
    del m; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

t0 = time.perf_counter()
run('B', STUDENT_DIR)
if os.path.isdir(E2_DIR): run('E2', E2_DIR)
for seed in SEEDS:
    run(f'S{seed}', os.path.join(SAVE_ROOT, f'seed{seed}_ep{EPOCHS}'))
print(f'판정 {len(queries):,}개 × {len(SC)} 모델, {time.perf_counter()-t0:.0f}초')

def series(x, scorer):
    # 점수기 이름의 첫 단어로 모델을 고른다: B1/B2 → 기준선, E2 → E2 원본, S42 등 → 시드 모델
    key = scorer.split()[0]
    S = SC['B'] if key in ('B1', 'B2') else SC[key]
    if '문장 끝' in scorer:
        return [(t, S[q]) for t, q in x['pts2'] if t in x['ends']]
    s = int(scorer.split('s=')[1].rstrip(')'))
    return [(t, S[q]) for t, q in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % s == 0]

In [ ]:
def calibrate(items, scorer, arl0):
    clean = [x for x in items if x['onset'] is None]
    n_tok = sum(x['n_tok'] for x in clean)
    sc = np.sort(np.concatenate([[s for _, s in series(x, scorer)] for x in clean]))[::-1]
    n_alarm = max(1, int(n_tok / arl0))
    return float(sc[n_alarm - 1]) + 1e-6 if n_alarm <= len(sc) else float(sc[-1])

def evaluate(items, scorer, thr):
    # 엄격 기준(주 지표, 선행 연구와 같은 정의): 응답의 **첫 경보**가 발생 θ 이후여야 탐지로 인정.
    #   θ 전에 경보가 울리면 그 응답은 오경보로 끝난 것(미탐지)으로 본다.
    # 관대 기준(참고): θ 전 경보는 무시하고 θ 이후 첫 경보를 탐지로 인정 — v1의 주 지표였으나 재현율이 부풀려짐.
    clean = [x for x in items if x['onset'] is None]
    hall = [x for x in items if x['onset'] is not None]
    alarms = sum(sum(s >= thr for _, s in series(x, scorer)) for x in clean)
    arl = sum(x['n_tok'] for x in clean) / max(alarms, 1)
    clean_flag = np.mean([any(s >= thr for _, s in series(x, scorer)) for x in clean])
    st_det, st_d, st_ch, cedd, le_det, pre = 0, [], [], [], 0, 0
    for x in hall:
        ser = series(x, scorer)
        first = next((t for t, s in ser if s >= thr), None)
        le_hit = next((t for t, s in ser if t >= x['onset'] and s >= thr), None)
        le_det += le_hit is not None
        pre += first is not None and first < x['onset']
        if first is not None and first >= x['onset']:
            st_det += 1; st_d.append(first - x['onset']); cedd.append(first - x['onset'])
            st_ch.append(x['te'][first] - x['ts'][x['onset']])
        else:
            cedd.append(x['n_tok'] - x['onset'])
    return {'realized_ARL0_dev': arl, 'clean_resp_flagged': clean_flag,
            'recall': st_det / len(hall), 'delay_tok_mean': np.mean(st_d) if st_d else np.nan,
            'delay_tok_median': np.median(st_d) if st_d else np.nan,
            'delay_chars_median': np.median(st_ch) if st_ch else np.nan,
            'censored_EDD_tok': np.mean(cedd), 'pre_onset_alarm_rate': pre / len(hall),
            'recall_lenient': le_det / len(hall),
            'calls_per_resp': np.mean([len(series(x, scorer)) for x in items])}

def calibrate_flag(items, scorer, rate):
    # 서비스 기준 보정: 환각 없는 응답 중 표시되는 비율이 rate 이하가 되는 임계값 (응답별 최대 점수의 분위수)
    mx = np.array([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    return float(np.quantile(mx, 1 - rate)) + 1e-6

## 9. 결과

In [ ]:
MODELS = ['B'] + (['E2'] if 'E2' in SC else []) + [f'S{s}' for s in SEEDS]
DEV_Q, DEV_Y = point_labels(dev_items)
rows = []
for mname in MODELS:
    names = (['B1 문장 끝', 'B2 앞부분 (s=4)', 'B2 앞부분 (s=2)'] if mname == 'B' else
             [f'{mname} 문장 끝', f'{mname} 앞부분 (s=4)', f'{mname} 앞부분 (s=2)'])
    ys, ss = [], []
    for x in dev_items:
        d = dict(x['pts2'])
        for sn in x['sents']:
            ys.append(sn['y']); ss.append(SC[mname][d[sn['t1']]])
    auc_full, auc_pref = roc_auc_score(ys, ss), roc_auc_score(DEV_Y, SC[mname][DEV_Q])
    for sc in names:
        r100 = evaluate(dev_items, sc, calibrate(val_items, sc, 100))
        rsv = evaluate(dev_items, sc, calibrate_flag(val_items, sc, 0.10))
        rows.append({'model': mname, 'scorer': sc, 'recall_ARL100': r100['recall'],
                     'delay_med_ARL100': r100['delay_tok_median'], 'delay_chars_ARL100': r100['delay_chars_median'],
                     'clean_flag_ARL100': r100['clean_resp_flagged'],
                     'recall_svc10': rsv['recall'], 'delay_med_svc10': rsv['delay_tok_median'],
                     'clean_flag_svc10': rsv['clean_resp_flagged'], 'auc_full': auc_full, 'auc_prefix': auc_pref})
res = pd.DataFrame(rows)
pd.set_option('display.width', 220)
print(res.round(3).to_string(index=False))

In [ ]:
# 시드 평균 ± 표준편차
seed_rows = res[res.model.str.match(r'S\d+')].copy()
seed_rows['kind'] = seed_rows.scorer.str.split(' ', n=1).str[1]
agg = seed_rows.groupby('kind')[['recall_ARL100', 'delay_med_ARL100', 'recall_svc10', 'delay_med_svc10',
                                  'auc_full', 'auc_prefix']].agg(['mean', 'std']).round(3)
print('=== 시드 3개 평균 ± 표준편차 ===')
print(agg.to_string())

## 10. 사전 기준 판정과 저장

In [ ]:
print('=== 안정성: 시드마다 E2 주 기준 (s=4 또는 s=2에서 재현율 ≥ 0.474 그리고 지연 ≤ 9토큰) ===')
per_seed = {}
for seed in SEEDS:
    g = res[res.model == f'S{seed}'].set_index('scorer')
    oks = []
    for s in (4, 2):
        r = g.loc[f'S{seed} 앞부분 (s={s})']
        oks.append(r.recall_ARL100 >= 0.474 and r.delay_med_ARL100 <= 9)
        print(f'  시드 {seed:4} s={s}: 재현율 {r.recall_ARL100:.3f}, 지연 {r.delay_med_ARL100:.1f} → {"충족" if oks[-1] else "미달"}')
    per_seed[seed] = any(oks)
stable = all(per_seed.values())
m = agg.loc['앞부분 (s=4)']
mean_ok = m[('recall_ARL100', 'mean')] >= 0.474 and m[('delay_med_ARL100', 'mean')] <= 9
print(f"\n안정성(시드 3개 모두 충족): {'충족' if stable else '미달'} — {per_seed}")
print(f"평균(s=4): 재현율 {m[('recall_ARL100','mean')]:.3f} ± {m[('recall_ARL100','std')]:.3f}, "
      f"지연 {m[('delay_med_ARL100','mean')]:.1f} ± {m[('delay_med_ARL100','std')]:.1f} → {'충족' if mean_ok else '미달'}")
ep2 = 0
for seed in SEEDS:
    h = pd.read_csv(os.path.join(SAVE_ROOT, f'seed{seed}_ep{EPOCHS}', 'hist.csv'))
    best = h.loc[h.val_prefix_auroc.idxmax()]
    ep2 += best.step > h.step.max() // 2
print(f'2에폭 효과: 두 번째 에폭에서 체크포인트가 선택된 시드 {ep2}/{len(SEEDS)}')
print('\n결론:', 'E2 결과는 시드에 안정적' if stable and mean_ok else 'E2 결과는 시드에 민감 — 중간보고에서는 시드 평균만 주장')

os.makedirs(os.path.join(OUT_DIR, 'e4lite'), exist_ok=True)
res.to_csv(os.path.join(OUT_DIR, 'e4lite', 'e4lite_results.csv'), index=False)
agg.to_csv(os.path.join(OUT_DIR, 'e4lite', 'e4lite_seed_agg.csv'))
hs = []
for seed in SEEDS:
    h = pd.read_csv(os.path.join(SAVE_ROOT, f'seed{seed}_ep{EPOCHS}', 'hist.csv')); h['seed'] = seed; hs.append(h)
pd.concat(hs).to_csv(os.path.join(OUT_DIR, 'e4lite', 'e4lite_train_hist.csv'), index=False)
print('저장:', os.path.join(OUT_DIR, 'e4lite'))